# LLM Inference Perf Lab (Colab)

Runs vLLM in this Colab runtime, then exercises it with the OpenAI-compatible client from `src/client.py`.

**Requirements**
- Colab runtime with a GPU (Runtime → Change runtime type → GPU)
- Run cells top to bottom (or Runtime → Run all)

Setup cells are adapted from `notebooks/vllm_setup.ipynb`. The first `vllm serve` run downloads `Qwen/Qwen2.5-1.5B-Instruct` from Hugging Face.

## 1. Check GPU

In [1]:
!nvidia-smi

Thu Jul 16 00:22:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install PyTorch, vLLM, and OpenAI client

In [2]:
!pip install -q uv
!pip install -q \
  "torch==2.9.0" \
  "torchvision==0.24.0" \
  "torchaudio==2.9.0" \
  --index-url https://download.pytorch.org/whl/cu128

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.9/26.9 MB 65.2 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 60.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 81.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.7/124.7 MB 122.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 900.9/900.9 MB 1.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 103.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 6.2 MB/s eta 0:00:00:00:0100:01


In [3]:
!pip install -q "vllm==0.11.2" openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.3/370.3 MB 1.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.0/355.0 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.0/183.0 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 82.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 117.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 98.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 7.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━

## 3. Verify install

In [4]:
import importlib.metadata as metadata
import torch

print("Installed vLLM package:", metadata.version("vllm"))
print("PyTorch:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Installed vLLM package: 0.11.2
PyTorch: 2.9.0+cu128
PyTorch CUDA build: 12.8
CUDA available: True
GPU: Tesla T4


In [5]:
import vllm

print("vLLM import successful")
print("vLLM version:", vllm.__version__)

vLLM import successful
vLLM version: 0.11.2


## 4. Start vLLM server (downloads model on first run)

Serves `Qwen/Qwen2.5-1.5B-Instruct` as `qwen2.5-1.5b` on port 8000. Logs go to `/content/vllm_server.log`.

In [18]:
%%bash

nohup vllm serve Qwen/Qwen2.5-1.5B-Instruct \
  --host 0.0.0.0 \
  --port 8000 \
  --served-model-name qwen2.5-1.5b \
  --dtype half \
  --max-model-len 2048 \
  --gpu-memory-utilization 0.85 \
  > /content/vllm_server.log 2>&1 &

echo $! > /content/vllm_server.pid
echo "Server process started with PID $(cat /content/vllm_server.pid)"

Server process started with PID 4093


## 5. Wait until the server is ready

First start can take several minutes while the model downloads and loads. This cell tails `/content/vllm_server.log` so you can watch download/load progress instead of a silent poll.

In [19]:
import os
import time
from pathlib import Path

import requests

url = "http://127.0.0.1:8000/v1/models"
log_path = Path("/content/vllm_server.log")
pid_path = Path("/content/vllm_server.pid")

pid = int(pid_path.read_text().strip())
log_offset = 0
started = time.monotonic()
poll_s = 2

print(f"Waiting for server (PID {pid}) to become ready...")
print(f"Tailing {log_path}\n")

while True:
    if log_path.exists():
        with log_path.open("r", errors="replace") as log_file:
            log_file.seek(log_offset)
            chunk = log_file.read()
            log_offset = log_file.tell()
        if chunk:
            print(chunk, end="" if chunk.endswith("\n") else "\n", flush=True)

    try:
        response = requests.get(url, timeout=2)
        if response.status_code == 200:
            elapsed_s = time.monotonic() - started
            print(f"\nServer is ready after {elapsed_s:.0f}s.")
            print(response.json())
            break
    except requests.RequestException:
        pass

    try:
        os.kill(pid, 0)
    except OSError as exc:
        print("\nServer process exited before becoming ready.")
        print("\nLast part of server log:\n")
        if log_path.exists():
            print(log_path.read_text(errors="replace")[-5000:])
        raise RuntimeError(f"vLLM server process {pid} is not running") from exc

    time.sleep(poll_s)


Waiting for server (PID 4093) to become ready...
Tailing /content/vllm_server.log

2026-07-16 00:34:12.687223: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO 07-16 00:34:22 [scheduler.py:216] Chunked prefill is enabled with max_num_batched_tokens=2048.
(APIServer pid=4093) INFO 07-16 00:34:22 [api_server.py:1977] vLLM API server version 0.11.2
(APIServer pid=4093) INFO 07-16 00:34:22 [utils.py:253] non-default args: {'model_tag': 'Qwen/Qwen2.5-1.5B-Instruct', 'host': '0.0.0.0', 'model': 'Qwen/Qwen2.5-1.5B-Instruct', 'dtype': 'half', 'max_model_len': 2048, 'served_model_name': ['qwen2.5-1.5b'], 'gpu_memory_utilization': 0.85}
(APIServer pid=4093) INFO 07-16 00:34:23 [model.py:631] Resolved architecture: Qwen2ForCausalLM
(APIServer pid=4093) 

## 6. Configure client

Defaults match `src/client.py` and the local vLLM server started above.

In [21]:
# @title Client settings
BASE_URL = "http://127.0.0.1:8000/v1"  # @param {type:"string"}
API_KEY = "not-needed"  # @param {type:"string"}
MODEL_NAME = "qwen2.5-1.5b"  # @param {type:"string"}
MAX_TOKENS = 100  # @param {type:"integer"}
TEMPERATURE = 0.0  # @param {type:"number"}
PROMPT = "What is LLM inference latency?"  # @param {type:"string"}

print(f"base_url={BASE_URL}")
print(f"model={MODEL_NAME}")

base_url=http://127.0.0.1:8000/v1
model=qwen2.5-1.5b


## 7. Create client

In [22]:
from openai import OpenAI

client = OpenAI(
    base_url=BASE_URL,
    api_key=API_KEY,
)

print("Client ready.")

Client ready.


## 8. Single request

Same call as `src/client.py`, with wall-clock latency.

In [23]:
import time

t0 = time.perf_counter()
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": PROMPT,
        },
    ],
    max_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
)
elapsed_s = time.perf_counter() - t0

content = response.choices[0].message.content
usage = response.usage

print(content)
print()
print(f"latency_s={elapsed_s:.3f}")
if usage is not None:
    print(f"prompt_tokens={usage.prompt_tokens}")
    print(f"completion_tokens={usage.completion_tokens}")
    print(f"total_tokens={usage.total_tokens}")
    if usage.completion_tokens and elapsed_s > 0:
        # E2E rate (includes prefill/scheduling/HTTP), not pure decode throughput.
        print(
            f"output_tokens_per_e2e_s={usage.completion_tokens / elapsed_s:.2f}"
        )

LLM inference latency refers to the time it takes for an artificial intelligence language model (LLM) to generate a response or output after receiving input data. This can be influenced by various factors such as:

1. Model complexity: More complex models may take longer to process and respond.

2. Data size: Larger datasets require more processing power and time to analyze.

3. Hardware limitations: The computational resources available on the server hosting the model will affect performance.

4. Network conditions: Slow internet connections

latency_s=3.543
prompt_tokens=36
completion_tokens=100
total_tokens=136
tokens_per_s=28.23


## 9. Streaming request (optional)

Measures time-to-first-token (TTFT) and end-to-end latency.

In [24]:
import time

t0 = time.perf_counter()
ttft_s = None
chunks: list[str] = []
usage = None

stream = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": PROMPT}],
    max_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
    stream=True,
    stream_options={"include_usage": True},
)

for event in stream:
    if getattr(event, "usage", None) is not None:
        usage = event.usage
    if not event.choices:
        continue
    delta = event.choices[0].delta.content
    if not delta:
        continue
    if ttft_s is None:
        ttft_s = time.perf_counter() - t0
    chunks.append(delta)
    print(delta, end="", flush=True)

elapsed_s = time.perf_counter() - t0
print()
print()
print(f"ttft_s={ttft_s:.3f}" if ttft_s is not None else "ttft_s=n/a")
print(f"e2e_latency_s={elapsed_s:.3f}")
print(f"output_chars={sum(len(c) for c in chunks)}")
if usage is not None:
    print(f"prompt_tokens={usage.prompt_tokens}")
    print(f"completion_tokens={usage.completion_tokens}")
    print(f"total_tokens={usage.total_tokens}")
    if usage.completion_tokens and elapsed_s > 0:
        print(
            f"output_tokens_per_e2e_s={usage.completion_tokens / elapsed_s:.2f}"
        )
else:
    print("usage=n/a (stream did not include usage)")


LLM inference latency refers to the time it takes for an artificial intelligence language model (LLM) to generate a response or output after receiving input data. This can be influenced by various factors such as:

1. Model complexity: More complex models may take longer to process and respond.

2. Data size: Larger datasets require more processing power and time to analyze.

3. Hardware limitations: The computational resources available on the server hosting the model will affect performance.

4. Network conditions: Slow internet connections

ttft_s=0.092
e2e_latency_s=1.966
output_chars=548


## 10. Quick latency sweep (optional)

Runs the same prompt several times. Each row records latency **and** actual `usage` token counts (do not assume `completion_tokens == max_tokens`). Writes `results/latency_sweep.csv`.

In [25]:
# @title Sweep settings
NUM_RUNS = 5  # @param {type:"integer"}
WARMUP_RUNS = 1  # @param {type:"integer"}

import csv
import statistics
import time
from pathlib import Path

sweep_rows: list[dict] = []
measured_latencies_s: list[float] = []
warmup_run_idx = 0
measured_run_idx = 0

for i in range(WARMUP_RUNS + NUM_RUNS):
    is_warmup = i < WARMUP_RUNS
    if is_warmup:
        warmup_run_idx += 1
        run_idx = warmup_run_idx
    else:
        measured_run_idx += 1
        run_idx = measured_run_idx

    t0 = time.perf_counter()
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": PROMPT}],
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
    )
    elapsed_s = time.perf_counter() - t0

    usage = response.usage
    prompt_tokens = usage.prompt_tokens if usage is not None else None
    completion_tokens = usage.completion_tokens if usage is not None else None
    total_tokens = usage.total_tokens if usage is not None else None
    output_tokens_per_s = (
        completion_tokens / elapsed_s
        if completion_tokens is not None and elapsed_s > 0
        else None
    )

    row = {
        "run": run_idx,
        "is_warmup": is_warmup,
        "latency_s": elapsed_s,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": total_tokens,
        "output_tokens_per_s": output_tokens_per_s,
    }
    sweep_rows.append(row)

    tokens_part = (
        f"prompt={prompt_tokens} completion={completion_tokens} total={total_tokens} "
        f"output_tok_per_e2e_s={output_tokens_per_s:.2f}"
        if output_tokens_per_s is not None
        else "usage=n/a"
    )
    print(
        f"{'warmup' if is_warmup else 'run'} {run_idx}: "
        f"{elapsed_s:.3f}s ({tokens_part})"
    )

    if not is_warmup:
        measured_latencies_s.append(elapsed_s)

results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)
csv_path = results_dir / "latency_sweep.csv"

with csv_path.open("w", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "run",
            "is_warmup",
            "latency_s",
            "prompt_tokens",
            "completion_tokens",
            "total_tokens",
            "output_tokens_per_s",
        ],
    )
    writer.writeheader()
    for row in sweep_rows:
        writer.writerow(
            {
                "run": row["run"],
                "is_warmup": str(row["is_warmup"]).lower(),
                "latency_s": f"{row['latency_s']:.3f}",
                "prompt_tokens": (
                    "" if row["prompt_tokens"] is None else row["prompt_tokens"]
                ),
                "completion_tokens": (
                    "" if row["completion_tokens"] is None else row["completion_tokens"]
                ),
                "total_tokens": (
                    "" if row["total_tokens"] is None else row["total_tokens"]
                ),
                "output_tokens_per_s": (
                    ""
                    if row["output_tokens_per_s"] is None
                    else f"{row['output_tokens_per_s']:.2f}"
                ),
            }
        )

print()
print(f"wrote {csv_path}")
print(f"n={len(measured_latencies_s)}")
print(f"mean_s={statistics.mean(measured_latencies_s):.3f}")
print(
    f"stdev_s={statistics.stdev(measured_latencies_s):.3f}"
    if len(measured_latencies_s) > 1
    else "stdev_s=n/a"
)
print(f"min_s={min(measured_latencies_s):.3f}")
print(f"max_s={max(measured_latencies_s):.3f}")

completion_counts = [
    row["completion_tokens"]
    for row in sweep_rows
    if not row["is_warmup"] and row["completion_tokens"] is not None
]
if completion_counts:
    print(
        f"completion_tokens measured runs: "
        f"min={min(completion_counts)} max={max(completion_counts)} "
        f"all_equal={len(set(completion_counts)) == 1}"
    )

warmup 1: 1.787s (prompt=36 completion=100 total=136 tok/s=55.97)
run 1: 1.732s (prompt=36 completion=100 total=136 tok/s=57.73)
run 2: 1.735s (prompt=36 completion=100 total=136 tok/s=57.63)
run 3: 1.750s (prompt=36 completion=100 total=136 tok/s=57.16)
run 4: 1.808s (prompt=36 completion=100 total=136 tok/s=55.32)
run 5: 1.790s (prompt=36 completion=100 total=136 tok/s=55.86)

wrote results/latency_sweep.csv
n=5
mean_s=1.763
stdev_s=0.034
min_s=1.732
max_s=1.808
completion_tokens measured runs: min=100 max=100 all_equal=True


## Notes

| Piece | Source |
|---|---|
| GPU check, PyTorch/vLLM install, serve, readiness wait | `notebooks/vllm_setup.ipynb` |
| OpenAI client + latency helpers | `src/client.py` |
| Model | `Qwen/Qwen2.5-1.5B-Instruct` served as `qwen2.5-1.5b` |
| Endpoint | `http://127.0.0.1:8000/v1` (same Colab runtime) |
| Sweep CSV | `results/latency_sweep.csv` (`run,is_warmup,latency_s,prompt_tokens,completion_tokens,total_tokens,output_tokens_per_s`) |
| Archived runs | `results/run_YYYY-MM-DD/` (`latency_raw.csv`, `summary.md`) |

Server log: `/content/vllm_server.log`  
Server PID: `/content/vllm_server.pid`
